In [ ]:
workspace_id = ""
workspace_name = ""
lakehouse_id = ""
source_path = ""
destination_directory_path = ""
action = ""
sdoh_copy= ""
source_destination_mapping_paths = []

StatementMeta(, c58f06ce-e803-406a-8f45-785428c69096, 5, Finished, Available, Finished)

In [ ]:
import sempy.fabric as fabric
from urllib.parse import urlparse

#Instantiate the client
client = fabric.FabricRestClient()

# Get lakehouse properties
response = client.get(f"v1/workspaces/{workspace_id}/lakehouses/{lakehouse_id}")
onelake_files_path = response.json()['properties']['oneLakeFilesPath']

# Parse the URL
parsed_url = urlparse(onelake_files_path)
domain = parsed_url.netloc

source_abfss_path = f"abfss://{workspace_id}@{domain}/{lakehouse_id}/{source_path}"
destination_abfss_path = f"abfss://{workspace_id}@{domain}/{lakehouse_id}/{destination_directory_path}"

print(source_abfss_path)
print(destination_abfss_path)

In [ ]:
if len(source_destination_mapping_paths) > 0:
    sdoh_copy_using_all = ""
    sdoh_source_abfss_path = ""
    sdoh_destination_abfss_path = ""
    for source_destination_mapping_path in source_destination_mapping_paths:
        source_abfss_path = f"abfss://{workspace_id}@{domain}/{lakehouse_id}/{source_destination_mapping_path['source_path']}"
        destination_abfss_path = f"abfss://{workspace_id}@{domain}/{lakehouse_id}/{source_destination_mapping_path['destination_directory_path']}"
        print(source_abfss_path)
        print(destination_abfss_path)
        if source_destination_mapping_path["module"] == "sdoh":
            sdoh_copy_using_all = source_destination_mapping_path["action"]
            sdoh_source_abfss_path = f"abfss://{workspace_id}@{domain}/{lakehouse_id}/{source_destination_mapping_path['source_path']}"
            sdoh_destination_abfss_path = f"abfss://{workspace_id}@{domain}/{lakehouse_id}/{source_destination_mapping_path['destination_directory_path']}"
            print(source_abfss_path)
        if action == "copy" and sdoh_copy == "":
            print("Copying files...")
            mssparkutils.fs.cp(source_abfss_path, destination_abfss_path, recurse=True)
        elif action == "move" and sdoh_copy == "":
            print("Moving files...")
            mssparkutils.fs.mv(source_abfss_path, destination_abfss_path)
    
    if sdoh_copy_using_all != "":
        sdoh_copy = sdoh_copy_using_all
        source_abfss_path = sdoh_source_abfss_path
        destination_abfss_path = sdoh_destination_abfss_path
        

In [ ]:
if sdoh_copy != "":
    
    def copy_source_files_and_folders(source_path, destination_path):
        # List the contents of the source directory
        source_contents = mssparkutils.fs.ls(source_path)
        
        # List the contents of the destination directory
        try:
            destination_contents = mssparkutils.fs.ls(destination_path)
            destination_files = {item.path.split('/')[-1]: item.path for item in destination_contents}
        except Exception as e:
            print(f"Destination path {destination_path} does not exist or is empty. Creating the path.")
            destination_files = {}
            mssparkutils.fs.mkdirs(destination_path)
        
        # Copy each item inside the source directory to the destination directory
        for item in source_contents:
            item_path = item.path
            item_name = item_path.split('/')[-1]
            destination_item_path = f"{destination_path}/{item_name}"
        
            if item.isDir:
                # Recursively copy the contents of the directory
                copy_source_files_and_folders(item_path, destination_item_path)
            else:
                if item_name in destination_files:
                    print(f"File already exists, skipping: {destination_item_path}")
                else:
                    print(f"Creating new file: {destination_item_path}")
                    mssparkutils.fs.cp(item_path, destination_item_path, recurse=True)
    
    sdoh_csv_data_path = f"abfss://{workspace_id}@{domain}/{lakehouse_id}/Files/SampleData/SDOH/CSV"
    sdoh_xlsx_data_path = f"abfss://{workspace_id}@{domain}/{lakehouse_id}/Files/SampleData/SDOH/XLSX"
        
    destination_path_csv = f"abfss://{workspace_id}@{domain}/{lakehouse_id}/Files/Ingest/SDOH/CSV"
    destination_path_xlsx = f"abfss://{workspace_id}@{domain}/{lakehouse_id}/Files/Ingest/SDOH/XLSX"
        
    # Copy the files along with their parent folders
    copy_source_files_and_folders(sdoh_csv_data_path, destination_path_csv)
    copy_source_files_and_folders(sdoh_xlsx_data_path, destination_path_xlsx)

elif action == "copy":
    print("Copying files...")
    mssparkutils.fs.cp(source_abfss_path, destination_abfss_path, recurse=True)
elif action == "move":
    print("Moving files...")
    mssparkutils.fs.mv(source_path, destination_abfss_path)

StatementMeta(, c58f06ce-e803-406a-8f45-785428c69096, 6, Finished, Available, Finished)

True